# Feature-Based Image Registration Example

This notebook demonstrates feature-based image registration using scikit-image for aligning time-lapse images of a beating heart.

We will:
1.  Load two example images.
2.  Detect features in both images.
3.  Match the detected features.
4.  Estimate the transformation.
5.  Apply the transformation to align the images.


## 1. Import Necessary Libraries

In [1]:
import numpy as np
try:
    from skimage import io
    from skimage.feature import ORB, match_descriptors
    from skimage.transform import estimate_transform, warp
except ImportError:
    print("Please install scikit-image. You can use: pip install scikit-image")
import matplotlib.pyplot as plt

Please install scikit-image. You can use: pip install scikit-image


## 2. Load Example Images

For demonstration purposes, we'll use two example images. Replace these with your actual time-lapse images.

In [2]:
import skimage.io as io
# Create dummy images
image1 = np.zeros((200, 200), dtype=np.float32)
image1[50:150, 50:150] = 1
image2 = np.zeros((200, 200), dtype=np.float32)
image2[60:160, 60:160] = 1

io.imsave("image1.png", (image1*255).astype(np.uint8))
io.imsave("image2.png", (image2*255).astype(np.uint8))

image1 = io.imread("image1.png", as_gray=True)
image2 = io.imread("image2.png", as_gray=True)

ModuleNotFoundError: No module named 'skimage'

## 3. Feature Detection

We use the ORB (Oriented FAST and Rotated BRIEF) feature detector, which is efficient and rotation-invariant.

In [ ]:
orb = ORB(n_keypoints=200)

orb.detect_and_extract(image1)
keypoints1 = orb.keypoints
descriptors1 = orb.descriptors

orb.detect_and_extract(image2)
keypoints2 = orb.keypoints
descriptors2 = orb.descriptors

## 4. Feature Matching

Match the descriptors between the two images.

In [ ]:
matches = match_descriptors(descriptors1, descriptors2, cross_check=True)

## 5. Estimate Transformation

Estimate the transformation matrix using the matched keypoints. We use a SimilarityTransform, which allows for rotation, translation, and scaling.

In [ ]:
src = keypoints2[matches[:, 1]][:, ::-1]
dst = keypoints1[matches[:, 0]][:, ::-1]

tform = estimate_transform('similarity', src, dst)

## 6. Apply Transformation

Apply the estimated transformation to the second image to align it with the first image.

In [ ]:
warped = warp(image2, tform.inverse, output_shape=image1.shape)
io.imsave("warped_image.png", (warped*255).astype(np.uint8))

## 7. Visualize the Results

Display the original images and the aligned image.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))
ax = axes.ravel()

ax[0].imshow(image1, cmap=plt.cm.gray)
ax[0].set_title("Image 1")

ax[1].imshow(image2, cmap=plt.cm.gray)
ax[1].set_title("Image 2")

ax[2].imshow(warped, cmap=plt.cm.gray)
ax[2].set_title("Warped Image 2")

for a in ax:
    a.set_axis_off()

plt.tight_layout()
plt.savefig("registration_result.png")
plt.show()

This is a basic example. For more complex cases, you might need to fine-tune the parameters of the feature detector, matcher, and transformation estimator. Also, non-rigid registration techniques might be needed for the beating heart data.